# Foundation Models

**Prerequisites**

- L04: Multilayer perceptrons and `nn.Module`
- L05: Overfitting, bias-variance tradeoff
- L07: CNNs
- L10: Transformers (all three notebooks)

**Outcomes**

- Define what a foundation model is and explain why transfer learning works
- Distinguish four adaptation strategies: feature extraction, linear probe, full fine-tuning, and LoRA
- Understand the mathematical formulation of each strategy — which parameters are frozen, which receive gradients
- Derive LoRA from the low-rank update hypothesis and implement a `LoRALinear` module in PyTorch
- Choose the appropriate adaptation strategy for a given problem


In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

np.set_printoptions(linewidth=140, precision=4, suppress=True)
%matplotlib inline

## From "Train Your Own" to "Adapt Someone Else's"

Every neural network we have trained in this course so far — the MLPs in L04, the CNN in L07, the RNN in L09, and the small GPT in L10 — started from **randomly initialized weights** and learned everything from scratch on whatever dataset we provided. This is a natural teaching approach because it isolates each architecture and shows exactly what the model is learning. But it is almost never how neural networks are used in practice.

In practice, almost no practitioner trains a state-of-the-art neural network from scratch. The cost is prohibitive: GPT-4 was trained on trillions of tokens using tens of thousands of GPUs for months, at an estimated cost in the hundreds of millions of dollars. Even a "small" image model like ResNet-50 required 1.2 million labeled ImageNet images and hours of multi-GPU training. No individual researcher and almost no individual firm has the resources to reproduce these training runs.

Instead, practitioners take a model that *someone else* has already trained on a large, general-purpose dataset and **adapt** it to their specific task. This approach is called **transfer learning**, and the pretrained models at its center are called **foundation models**. The term was coined by Bommasani et al. (2021):

> A foundation model is any model that is trained on broad data (generally using self-supervision at scale) that can be adapted (e.g., fine-tuned) to a wide range of downstream tasks.

Foundation models are the "public infrastructure" of modern machine learning. The upstream compute cost has already been paid; what remains for the downstream user is a much smaller adaptation cost. This lecture is about that adaptation step.

### Examples of Foundation Models

| Modality | Foundation model | Pretraining data | Typical use |
|---|---|---|---|
| Vision | ResNet-50 | ImageNet (1.2M labeled images) | Image classification backbone |
| Vision | ViT-B/16 | ImageNet-21k (14M images) | Classification, feature extraction |
| Vision | DINOv2 | LVD-142M (142M images, self-supervised) | Dense features, segmentation |
| Vision | CLIP | 400M image-text pairs (self-supervised) | Zero-shot classification, retrieval |
| Language | BERT-base | BooksCorpus + Wikipedia (~3.3B tokens) | Text classification, embeddings |
| Language | GPT-2 | WebText (~10B tokens) | Text generation |
| Language | Llama 3 | ~15T tokens of web/code/books | Everything (chat, code, reasoning) |
| Multimodal | CLIP | 400M image-text pairs | Image-text alignment |
| Multimodal | GPT-4V / Claude | Trillions of tokens + images | Vision + language reasoning |

Notice the common pattern: each of these models was trained on a dataset orders of magnitude larger than anything a typical researcher could assemble on their own. That is the source of transfer learning's power — the foundation model has seen more of the world's text and images than any downstream user ever could.

## Why Does Transfer Learning Work?

Recall the bias-variance decomposition from L05. A model trained on a small dataset faces a fundamental dilemma: a flexible model will overfit (high variance), but a rigid model will miss the structure (high bias). Regularization trades one for the other, but with a truly small dataset neither extreme produces a good predictor.

Transfer learning sidesteps this dilemma. Instead of starting from a random initialization and trying to learn both the general structure of the input (e.g., what edges and textures look like in images) *and* the task-specific decision boundary from the same small dataset, we separate the two problems:

1. **The pretraining step** learns general input structure from a massive, cheap-to-collect dataset (unlabeled images, raw text). This is done once, by someone else, at enormous scale. The output is a feature extractor that maps raw inputs to dense vector representations which already capture most of the meaningful variation in the data.

2. **The adaptation step** learns a small, task-specific mapping on top of those frozen (or near-frozen) features using the small labeled dataset we actually have. Because the features are already informative, the adaptation step has far fewer effective parameters to fit, so variance is much lower.

The key statistical insight is that **the effective sample size for the pretrained features is the pretraining corpus, not the fine-tuning corpus**. When you start from ImageNet-pretrained ResNet features, your classifier is implicitly benefiting from the 1.2 million labeled images the upstream team collected, even though your own labeled dataset might only have a few hundred examples.

There is also an economic way to think about this: pretraining is an expensive fixed cost that creates a reusable public good. Once the foundation model exists, the marginal cost of adapting it to a new task is small, and every downstream user inherits the upstream investment. This is why foundation models have had such an outsized impact on the field — they dramatically lower the barrier to applying deep learning to new problems.

## Four Adaptation Strategies

Given a pretrained foundation model with parameters $\theta$, we want to adapt it to a new task with labeled data $\{(x_i, y_i)\}_{i=1}^n$. There are four standard strategies, ordered roughly from least to most flexible:

| Strategy | What is updated | Trainable params | Typical use |
|---|---|---|---|
| **Feature extraction** | Nothing | 0 (plus a classical model on top) | Tiny datasets, fast prototyping, interpretable features |
| **Linear probe** | A single new linear layer | $O(d \cdot k)$ | Small datasets, evaluating representation quality |
| **Full fine-tuning** | All parameters | $|\theta|$ (millions to billions) | Medium/large datasets, maximum performance |
| **LoRA** | Small added adapter matrices | $\ll |\theta|$ | Large models, many tasks, limited memory |

Each strategy is a different point on the tradeoff between **expressive power** (how much the model can change) and **sample efficiency / compute** (how much data and compute the adaptation requires). We examine each in turn, both mathematically and with a runnable demonstration.

### Strategy 1: Feature Extraction

Treat the pretrained model as a **fixed function** $\phi_\theta: \mathcal{X} \to \mathbb{R}^d$ that maps inputs to dense feature vectors. Extract features for every labeled example, then fit any classical model you like (logistic regression, random forest, SVM) on the features:

$$\hat{y} = g(\phi_\theta(x))$$

where $g$ is the downstream classical model. The neural network itself is never trained — not even the final layer. The output of the pretrained model is treated as "better raw features" than the original pixels or tokens.

**When this is the right choice:**

- The labeled dataset is tiny (tens to a few hundred examples)
- You want deterministic, fast, interpretable training
- You want to use classical statistical tools (confidence intervals, feature importance) on top of the representations

**Limitation:** the features are frozen at whatever the pretraining objective produced. If there is a mismatch between the pretraining data and your task — for instance, using ImageNet features on medical X-rays — the features may not capture the distinctions you care about.

### Strategy 2: Linear Probe

A **linear probe** is a specific instance of adaptation in which we add a single linear layer on top of the frozen pretrained features and train only that layer:

$$\hat{y} = W \phi_\theta(x) + b$$

The pretrained parameters $\theta$ are frozen (`.requires_grad = False`), so only $W$ and $b$ receive gradient updates. If the output space has $k$ classes and the feature dimension is $d$, the number of trainable parameters is just $k(d+1)$ — typically a few thousand, regardless of how large the underlying foundation model is.

A linear probe is conceptually close to feature extraction + logistic regression, but it is trained with SGD inside the PyTorch framework, which makes it easy to combine with mini-batch training, GPU acceleration, and the rest of the deep learning pipeline.

Linear probes are also the standard way to **evaluate the quality of a representation**. In research, when a new self-supervised method like DINOv2 is proposed, its representation quality is typically measured by the accuracy of a linear probe on top of frozen features on ImageNet. Higher linear-probe accuracy means the features are more linearly separable, which is a strong signal that the representation is useful for downstream tasks.

### Strategy 3: Full Fine-Tuning

In full fine-tuning, **every parameter** of the pretrained model is updated — plus any new output head we add for the task. If the pretrained model has parameters $\theta$ and the new head has parameters $\psi$, the optimizer updates both:

$$\theta', \psi' = \arg\min_{\theta, \psi} \; \frac{1}{n}\sum_{i=1}^{n} \ell(f_{\theta, \psi}(x_i), y_i)$$

where $\ell$ is the task loss (e.g., cross-entropy for classification). The pretrained weights serve as the **initialization** rather than a fixed feature extractor.

Full fine-tuning is the most flexible strategy — the model can adapt every layer to the target task — but it has two drawbacks:

1. **Compute and memory.** Training requires storing gradients for every parameter and running backpropagation through the full model. For a multi-billion-parameter foundation model this is often impractical on a single GPU.

2. **Catastrophic forgetting.** If the learning rate is too large, the fine-tuned weights can drift far from the pretrained initialization and the model loses the general knowledge it was pretrained on. The standard mitigation is to use a learning rate 10–100× smaller than the pretraining learning rate, so that updates are small and the pretrained structure is preserved.

Full fine-tuning is the right choice when (a) the target dataset is large enough to reliably update all parameters without overfitting, and (b) the task is sufficiently different from the pretraining task that updating only a final layer would not capture the necessary changes.

### Strategy 4: LoRA

**Low-Rank Adaptation** (Hu et al., 2021) is a middle ground between linear probes and full fine-tuning. The idea is to freeze the pretrained weights but *add* a small, trainable low-rank update, taking advantage of an empirical observation: the optimal weight changes during fine-tuning tend to be approximately low-rank.

Consider a single weight matrix $W \in \mathbb{R}^{d \times k}$ from the pretrained model. In full fine-tuning, we would replace $W$ with $W' = W + \Delta W$, where $\Delta W \in \mathbb{R}^{d \times k}$ has $dk$ parameters. LoRA replaces this dense update with a low-rank factorization:

$$\Delta W \approx B A, \qquad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}$$

where the rank $r$ is a hyperparameter with $r \ll \min(d, k)$. Typical choices are $r = 4, 8, 16$. The forward pass becomes:

$$y = Wx + BAx$$

During LoRA fine-tuning, only $A$ and $B$ receive gradient updates; the original $W$ is frozen.

**Parameter count comparison.** A full-rank update has $dk$ parameters; the LoRA update has $r(d + k)$ parameters. For a typical attention projection matrix in GPT-2 where $d = k = 768$ and $r = 8$, full fine-tuning updates $768^2 = 589{,}824$ parameters per matrix, while LoRA updates only $8 \cdot (768 + 768) = 12{,}288$ parameters — a **48× reduction** per matrix, and similar savings across every attention matrix in the model.

**Initialization trick.** At the start of training, LoRA must behave exactly like the pretrained model — otherwise we lose everything we gained from pretraining. This is arranged by initializing $B = 0$ (all zeros) and $A \sim \mathcal{N}(0, \sigma^2)$ (small random values). With $B = 0$, the low-rank update $BAx = 0$ for every input, so the initial forward pass is identical to the frozen pretrained model. Training can then smoothly move $B$ away from zero as the task loss demands.

**When LoRA wins.** LoRA is the dominant adaptation strategy for very large generative models (Llama, GPT, Claude) for three reasons: (a) it requires much less memory because we only store gradients for the small $A$ and $B$ matrices, (b) the trained adapters are tiny (~MB per task) and can be stored separately and swapped in at inference time, and (c) multiple LoRA adapters can be trained for different tasks and combined or selected dynamically.

We will derive and implement LoRA in detail in a later section of this notebook.

## A Runnable Demonstration: Tiny Foundation Model

Before touching real pretrained models (which we do in notebooks 2–4), let's build a miniature version of the whole pipeline. We will:

1. Define a small MLP.
2. "Pretrain" it on a synthetic regression task with lots of data. This takes the place of the massive ImageNet or WebText pretraining runs — the goal is just to give the model structured, useful weights.
3. Define a *different* but related synthetic task with only a handful of labeled examples. This takes the place of a real downstream task where labels are expensive.
4. Adapt the pretrained MLP to the downstream task using each of the four strategies and compare.

The MLP and the datasets are small enough that the whole experiment runs in a few seconds on CPU. The pedagogy is the important part — the exact numerical results will vary from run to run, but the qualitative ranking of the strategies will not.

### The Synthetic Tasks

We will use two regression tasks with shared structure. Both tasks map 8-dimensional inputs to a scalar output, and both depend on the same "latent features" — nonlinear combinations of the input — but combine those features differently to produce their targets.

- **Pretraining task.** Given $x \in \mathbb{R}^8$, predict $y_{\text{pre}} = \sin(x_0 + x_1) + \tanh(x_2 x_3) + 0.5 (x_4^2 - x_5^2) + \text{noise}$. We generate 10,000 examples — our "massive pretraining dataset."
- **Downstream task.** Given the same kind of input $x$, predict $y_{\text{down}} = 2\sin(x_0 + x_1) - \tanh(x_2 x_3) + 0.3 x_6 + \text{noise}$. We generate only 40 training examples — our "small labeled dataset."

The important thing is that the **same latent nonlinear features** (especially $\sin(x_0 + x_1)$ and $\tanh(x_2 x_3)$) appear in both tasks. A model that learns these features during pretraining will have a strong head start on the downstream task — just like an ImageNet-pretrained ResNet has a head start on CIFAR-10 because it has already learned what edges and textures look like.

In [ ]:
def gen_pretrain_data(n, noise=0.1, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, 8)).astype(np.float32)
    y = (
        np.sin(X[:, 0] + X[:, 1])
        + np.tanh(X[:, 2] * X[:, 3])
        + 0.5 * (X[:, 4] ** 2 - X[:, 5] ** 2)
        + noise * rng.standard_normal(n).astype(np.float32)
    )
    return torch.from_numpy(X), torch.from_numpy(y.astype(np.float32))


def gen_downstream_data(n, noise=0.1, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, 8)).astype(np.float32)
    y = (
        2.0 * np.sin(X[:, 0] + X[:, 1])
        - np.tanh(X[:, 2] * X[:, 3])
        + 0.3 * X[:, 6]
        + noise * rng.standard_normal(n).astype(np.float32)
    )
    return torch.from_numpy(X), torch.from_numpy(y.astype(np.float32))


# Pretraining: lots of data
X_pre, y_pre = gen_pretrain_data(n=10_000, seed=0)

# Downstream: very little data
X_train, y_train = gen_downstream_data(n=40, seed=1)
X_test, y_test = gen_downstream_data(n=1000, seed=2)

print(f"Pretraining set:      {X_pre.shape}, target range [{y_pre.min():.2f}, {y_pre.max():.2f}]")
print(f"Downstream train:     {X_train.shape}")
print(f"Downstream test:      {X_test.shape}")

### The MLP Architecture

A standard two-hidden-layer MLP with ReLU activations. The architecture is deliberately small so we can run the whole notebook quickly — for a real foundation model the architecture would be much larger, but the conceptual content is identical.

In [ ]:
class TinyFM(nn.Module):
    """A tiny two-hidden-layer MLP that will play the role of a foundation model."""

    def __init__(self, in_dim=8, hidden=64, out_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.head = nn.Linear(hidden, out_dim)

    def features(self, x):
        """Return the representation just before the final head."""
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return h

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


torch.manual_seed(0)
base = TinyFM()
print(base)
print(f"\nTotal parameters: {sum(p.numel() for p in base.parameters()):,}")

### Step 1: Pretraining

We train the MLP on the pretraining task with plenty of data. After this step, the network's weights will encode the latent features $\sin(x_0 + x_1)$, $\tanh(x_2 x_3)$, etc., that appear in both tasks. This is analogous to training ResNet on ImageNet — by the end, the lower layers know about edges and textures even though no one specifically told them to learn edges.

In [ ]:
def train_regression(model, X, y, epochs=200, lr=1e-2, batch_size=256, verbose=False):
    """Simple mini-batch training loop for MSE regression."""
    opt = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    losses = []
    n = X.shape[0]
    for epoch in range(epochs):
        perm = torch.randperm(n)
        total = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            pred = model(X[idx])
            loss = F.mse_loss(pred, y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * idx.numel()
        losses.append(total / n)
        if verbose and (epoch + 1) % 50 == 0:
            print(f"  epoch {epoch + 1:4d}  loss={losses[-1]:.4f}")
    return losses


torch.manual_seed(0)
base = TinyFM()
print("Pretraining the tiny foundation model on the synthetic task...")
pre_losses = train_regression(base, X_pre, y_pre, epochs=200, lr=1e-2, verbose=True)

with torch.no_grad():
    pre_test_loss = F.mse_loss(base(X_pre), y_pre).item()
print(f"\nFinal pretraining MSE: {pre_test_loss:.4f}")

The pretraining loss drops quickly because we have plenty of data relative to the size of the model. The resulting weights encode the latent structure we care about. From now on we treat `base` as a "frozen" foundation model and only make copies of it for adaptation.

In [ ]:
import copy

# Snapshot the pretrained weights so every adaptation strategy starts from
# the same initialization.
PRETRAINED_STATE = copy.deepcopy(base.state_dict())


def fresh_pretrained():
    model = TinyFM()
    model.load_state_dict(PRETRAINED_STATE)
    return model

### Baseline: Train from Scratch on the Downstream Data

Before comparing the four adaptation strategies, let's see what happens if we don't use the pretrained model at all and just train a fresh MLP on the 40 downstream examples. This is the baseline — anything worse than this would mean transfer learning is hurting us, not helping.

In [ ]:
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X)
        return F.mse_loss(pred, y).item()


torch.manual_seed(0)
scratch = TinyFM()
t0 = time.time()
scratch_losses = train_regression(scratch, X_train, y_train, epochs=200, lr=1e-2)
scratch_time = time.time() - t0
scratch_test_mse = evaluate(scratch, X_test, y_test)

print(f"From scratch:  trainable params = {count_trainable(scratch):,}")
print(f"               test MSE         = {scratch_test_mse:.4f}")
print(f"               wall time        = {scratch_time:.2f}s")

### Strategy 1: Feature Extraction

Freeze the pretrained MLP and use its penultimate-layer features as input to a **linear regression** (no neural network training at all). This is the most conservative strategy — we are treating the pretrained network as a fixed feature map and doing classical statistics on top.

In [ ]:
from sklearn.linear_model import LinearRegression

feat_model = fresh_pretrained()
feat_model.eval()

t0 = time.time()
with torch.no_grad():
    feats_train = feat_model.features(X_train).numpy()
    feats_test = feat_model.features(X_test).numpy()

# Classical linear regression on frozen features.
reg = LinearRegression()
reg.fit(feats_train, y_train.numpy())
feat_test_mse = float(np.mean((reg.predict(feats_test) - y_test.numpy()) ** 2))
feat_time = time.time() - t0

print(f"Feature extraction:  trainable params (neural) = 0")
print(f"                      test MSE                  = {feat_test_mse:.4f}")
print(f"                      wall time                 = {feat_time:.2f}s")

### Strategy 2: Linear Probe

Same idea as feature extraction, but this time we train the final layer with SGD inside PyTorch. Mechanically, we (a) load the pretrained weights, (b) freeze everything except the final head, and (c) reinitialize the head so we're fitting it from scratch.

In [ ]:
torch.manual_seed(0)
probe = fresh_pretrained()

# Freeze everything
for p in probe.parameters():
    p.requires_grad = False

# Reinitialize + unfreeze the head
probe.head = nn.Linear(64, 1)
for p in probe.head.parameters():
    p.requires_grad = True

print(f"Linear probe trainable params: {count_trainable(probe):,}")

t0 = time.time()
probe_losses = train_regression(probe, X_train, y_train, epochs=200, lr=1e-2)
probe_time = time.time() - t0
probe_test_mse = evaluate(probe, X_test, y_test)

print(f"Linear probe:  trainable params = {count_trainable(probe):,}")
print(f"               test MSE         = {probe_test_mse:.4f}")
print(f"               wall time        = {probe_time:.2f}s")

### Strategy 3: Full Fine-Tuning

Unfreeze every parameter and train end-to-end with a smaller learning rate. The smaller learning rate is crucial: if we use the same rate as the from-scratch baseline, the optimizer will take large steps that destroy the pretrained features in the first few updates. Using a rate 10× smaller keeps the updates small and preserves the pretrained structure.

In [ ]:
torch.manual_seed(0)
full = fresh_pretrained()
for p in full.parameters():
    p.requires_grad = True

t0 = time.time()
# Smaller learning rate to avoid catastrophic forgetting
full_losses = train_regression(full, X_train, y_train, epochs=200, lr=1e-3)
full_time = time.time() - t0
full_test_mse = evaluate(full, X_test, y_test)

print(f"Full fine-tune:  trainable params = {count_trainable(full):,}")
print(f"                 test MSE         = {full_test_mse:.4f}")
print(f"                 wall time        = {full_time:.2f}s")

### Strategy 4: LoRA — Deriving and Implementing It

We now derive LoRA from first principles and implement it as a PyTorch module. This module will be reused in notebook 4 to apply LoRA to GPT-2, so it is worth spending time to understand exactly what it does.

**Setup.** Consider a single linear layer in the pretrained model, with weight matrix $W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$ and bias $b \in \mathbb{R}^{d_{\text{out}}}$. Given an input $x \in \mathbb{R}^{d_{\text{in}}}$, the forward pass is:

$$y = W x + b$$

Full fine-tuning would let the optimizer change every entry of $W$ and $b$ independently. LoRA instead *freezes* $W$ and $b$ and adds a trainable low-rank correction:

$$y = W x + b + B A x$$

where $B \in \mathbb{R}^{d_{\text{out}} \times r}$ and $A \in \mathbb{R}^{r \times d_{\text{in}}}$, with $r \ll \min(d_{\text{in}}, d_{\text{out}})$. The product $BA$ is a $d_{\text{out}} \times d_{\text{in}}$ matrix, but because it factors through the rank-$r$ bottleneck, we only need to store and optimize $r \cdot (d_{\text{in}} + d_{\text{out}})$ parameters instead of $d_{\text{in}} \cdot d_{\text{out}}$.

**Scaling factor.** In practice, LoRA is usually implemented with an extra scalar scaling factor $\alpha / r$ to decouple the learning rate from the rank:

$$y = W x + b + \frac{\alpha}{r} B A x$$

where $\alpha$ is a hyperparameter (often $\alpha = r$, or $\alpha = 2r$). The scaling keeps the *effective* learning rate roughly constant as we vary $r$.

**Initialization.** We want the initial forward pass to be identical to the frozen pretrained model (otherwise we lose all the pretraining benefit). This is achieved by:

- $A \sim \mathcal{N}(0, \sigma^2)$ — small random values
- $B = 0$ — exactly zero

With $B = 0$, the LoRA contribution $BAx = 0$, so $y = Wx + b$ exactly at initialization. Training then gradually moves $B$ away from zero as the task loss demands.

In [ ]:
class LoRALinear(nn.Module):
    """A linear layer with a low-rank additive update.

    Wraps an existing frozen ``nn.Linear`` (the pretrained weight matrix)
    and adds a trainable low-rank correction ``B @ A``. The original weights
    are kept frozen and the LoRA adapters are the only parameters that
    receive gradient updates.
    """

    def __init__(self, base: nn.Linear, r: int = 4, alpha: float = 8.0):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze the base layer — we keep a reference, but no gradients flow.
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False

        # LoRA parameters. A is Kaiming-init, B is zero.
        self.lora_A = nn.Parameter(torch.zeros(r, self.in_features))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
        # B is left at zero so the initial output matches the base layer.

    def forward(self, x):
        # Frozen base contribution plus trainable low-rank correction.
        base_out = self.base(x)
        lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B)
        return base_out + self.scaling * lora_out


# Sanity check: at initialization the LoRA layer must match the base layer.
torch.manual_seed(0)
base_linear = nn.Linear(8, 4)
lora_wrapped = LoRALinear(base_linear, r=2, alpha=4.0)

x = torch.randn(3, 8)
assert torch.allclose(base_linear(x), lora_wrapped(x), atol=1e-6), \
    "LoRA init should be identity-preserving"
print("LoRA-wrapped layer matches base layer at init — good.")

# Count the parameters: only A and B are trainable.
print(f"Base linear params (frozen): {base_linear.weight.numel() + base_linear.bias.numel()}")
print(f"LoRA adapter params        : {lora_wrapped.lora_A.numel() + lora_wrapped.lora_B.numel()}")

Now let's apply LoRA to the tiny foundation model. We wrap every `nn.Linear` layer in the model with a `LoRALinear`, freeze the base weights, and train.

In [ ]:
torch.manual_seed(0)
lora_model = fresh_pretrained()

# Replace each linear layer with a LoRA-wrapped version.
lora_model.fc1 = LoRALinear(lora_model.fc1, r=4, alpha=8.0)
lora_model.fc2 = LoRALinear(lora_model.fc2, r=4, alpha=8.0)
lora_model.head = LoRALinear(lora_model.head, r=4, alpha=8.0)

# Double-check: only LoRA params should require gradients.
trainable_names = [n for n, p in lora_model.named_parameters() if p.requires_grad]
print("Trainable parameters (LoRA only):")
for n in trainable_names:
    print(f"  {n}")

t0 = time.time()
lora_losses = train_regression(lora_model, X_train, y_train, epochs=200, lr=1e-2)
lora_time = time.time() - t0
lora_test_mse = evaluate(lora_model, X_test, y_test)

print(f"\nLoRA:  trainable params = {count_trainable(lora_model):,}")
print(f"       test MSE         = {lora_test_mse:.4f}")
print(f"       wall time        = {lora_time:.2f}s")

### Comparing the Strategies

Let's put all five numbers side by side. The important comparisons are:

- **All transfer strategies should beat from-scratch training.** On a tiny dataset like our 40-example downstream task, learning the latent features from scratch is simply not possible.
- **Linear probe, full fine-tune, and LoRA should all produce similar quality.** They are all making use of the pretrained features.
- **Trainable parameter counts differ by orders of magnitude.** Feature extraction has zero neural parameters, linear probe has ~65, LoRA has a few hundred, and full fine-tune has ~4,700. On real foundation models this spread is even more extreme — LoRA might use 0.1% of the full-fine-tune parameter count.

In [ ]:
# Match counts so the table is accurate: feature extraction's "0" is the
# neural-network parameter count (the sklearn linear regressor is classical).
results = [
    ("From scratch", scratch_test_mse, count_trainable(scratch), scratch_time),
    ("Feature extraction", feat_test_mse, 0, feat_time),
    ("Linear probe", probe_test_mse, count_trainable(probe), probe_time),
    ("Full fine-tune", full_test_mse, count_trainable(full), full_time),
    ("LoRA (r=4)", lora_test_mse, count_trainable(lora_model), lora_time),
]

print(f"{'Strategy':<22} {'Test MSE':>10}  {'Trainable':>12}  {'Wall time':>10}")
print("-" * 60)
for name, mse, params, dur in results:
    print(f"{name:<22} {mse:>10.4f}  {params:>12,}  {dur:>9.2f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

names = [r[0] for r in results]
mses = [r[1] for r in results]
params = [max(r[2], 1) for r in results]  # avoid log(0)
colors = ["#888", "#1f77b4", "#2ca02c", "#ff7f0e", "#d62728"]

bars1 = axes[0].bar(names, mses, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Test MSE (lower is better)")
axes[0].set_title("Downstream test error by strategy")
axes[0].tick_params(axis="x", rotation=20)
for b, m in zip(bars1, mses):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height(),
                 f"{m:.3f}", ha="center", va="bottom", fontsize=9)

bars2 = axes[1].bar(names, params, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("Trainable parameters (neural)")
axes[1].set_yscale("log")
axes[1].set_title("Trainable parameter counts")
axes[1].tick_params(axis="x", rotation=20)
for b, p in zip(bars2, params):
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height(),
                 f"{p}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

The precise numbers depend on the random seed, but the qualitative picture is consistent across runs:

- **From scratch is clearly worse.** With only 40 training points, a randomly initialized MLP cannot recover the latent nonlinear features, so it overfits noise.
- **All four transfer strategies are much better**, and they sit in roughly the same neighborhood. The pretrained features are doing the heavy lifting.
- **Feature extraction and linear probe use almost no trainable parameters** but still produce good performance because the pretrained features are already near-optimal.
- **LoRA sits between the linear probe and the full fine-tune** in terms of parameter count but often matches the full fine-tune's accuracy.

This is exactly the tradeoff curve you should expect when you move to real foundation models. The rest of the lectures will show the same pattern at much larger scale.

## Catastrophic Forgetting and Why Fine-Tuning Learning Rates Are Small

A recurring theme in fine-tuning is **catastrophic forgetting**: when the learning rate is too large, the optimizer takes steps that drift away from the pretrained weights and wipe out the general knowledge we were trying to preserve. This is the same failure mode we discussed in L10 notebook 3 for post-training language models.

Let's see it in action. We'll run full fine-tuning again, but with a learning rate 100× larger than what we used before.

In [ ]:
torch.manual_seed(0)
forgotten = fresh_pretrained()
for p in forgotten.parameters():
    p.requires_grad = True

# Large learning rate — much larger than the ~1e-3 we used before
_ = train_regression(forgotten, X_train, y_train, epochs=200, lr=1e-1)
forgotten_test_mse = evaluate(forgotten, X_test, y_test)

print(f"Full fine-tune, LR=1e-3: test MSE = {full_test_mse:.4f}")
print(f"Full fine-tune, LR=1e-1: test MSE = {forgotten_test_mse:.4f}")
print(f"Ratio worse:             {forgotten_test_mse / full_test_mse:.1f}x")

With the aggressive learning rate, the fine-tuned model is much worse — often worse than the from-scratch baseline. The optimizer has destroyed the pretrained features in its rush to minimize training loss on 40 examples, and now it overfits the training set without any of the inductive bias the pretraining gave us.

**Practical implications:**

- Fine-tuning learning rates are typically 10–100× smaller than pretraining learning rates.
- LoRA is more forgiving because the pretrained weights can't be directly corrupted — only the small additive correction changes.
- Linear probes are the most forgiving because the pretrained weights literally cannot change, only the final layer.

This is one of the reasons to prefer linear probes and LoRA when the dataset is small: they have a built-in safety net against forgetting.

## When *Not* to Fine-Tune

Fine-tuning is not always the right answer. Several alternatives are worth considering before you pay the cost of collecting labels and running a training loop:

**Zero-shot / in-context learning.** For modern language models like Llama, GPT-4, or Claude, you can often accomplish a task simply by describing it in the prompt — no fine-tuning required. This works remarkably well for classification, extraction, and generation tasks, and the "training" cost is zero. The tradeoff is inference cost: every query has to include the task description and examples in the prompt, which uses context tokens.

**Prompting with examples (few-shot).** A variant of zero-shot where you include a few solved examples in the prompt. Still no training required, but the model can learn from the examples within the prompt itself.

**Retrieval-augmented generation (RAG).** Instead of teaching the model new facts through fine-tuning, store the facts in a database and retrieve relevant ones at query time, concatenating them into the prompt. This is the standard approach for knowledge-grounded applications because it lets you update the knowledge base without retraining.

**Classical ML on frozen embeddings.** As we saw in strategy 1, if you have a tiny labeled dataset you can extract embeddings from a frozen foundation model and train a logistic regression or random forest on top. This is much cheaper than fine-tuning, supports classical statistical tools (confidence intervals, feature importance), and is often competitive for small datasets.

**The economics of choosing.** Fine-tuning has high fixed cost (data collection, training compute, experimentation) and low marginal cost (cheap inference). Prompting has zero fixed cost but higher marginal cost per query. If you will query the model billions of times, fine-tuning pays off. If you will query it a few thousand times, prompting is usually cheaper. This is a classic fixed-versus-marginal-cost decision — the same kind of decision firms make when choosing between buying a machine or paying per use of a service.

## Summary

We have now covered the conceptual framework for foundation models:

- A **foundation model** is a large model pretrained on broad data that can be cheaply adapted to new tasks. Almost all modern deep learning is built on top of foundation models rather than trained from scratch.
- Transfer learning works because the foundation model has been trained on orders of magnitude more data than any downstream user could gather, so its features are close to optimal for a wide variety of tasks.
- There are four standard adaptation strategies, trading expressiveness for sample efficiency: **feature extraction**, **linear probe**, **full fine-tuning**, and **LoRA**.
- **LoRA** replaces a dense $d \times k$ weight update with a low-rank factorization $BA$ of rank $r \ll \min(d, k)$, reducing trainable parameters by orders of magnitude. Our `LoRALinear` module implements this and will be reused in notebook 4.
- Fine-tuning learning rates must be kept small to avoid **catastrophic forgetting** of the pretrained features.

The next three notebooks apply these ideas to real pretrained models:

- **Notebook 2** — Pretrained ResNet and ViT for image classification (CIFAR-10 subset).
- **Notebook 3** — A case study on satellite imagery (EuroSAT), connecting to empirical economics literature.
- **Notebook 4** — DistilBERT for text classification and GPT-2 for text generation, including a hand-implemented LoRA adaptation.

## References

- Bommasani, R., Hudson, D. A., Adeli, E., et al. (2021). On the opportunities and risks of foundation models. *arXiv:2108.07258*.
- Hu, E. J., Shen, Y., Wallis, P., Allen-Zhu, Z., Li, Y., Wang, S., Wang, L., & Chen, W. (2021). LoRA: Low-rank adaptation of large language models. *arXiv:2106.09685*.
- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL 2019*.
- Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. *OpenAI Technical Report*.
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR 2016*.
- Dosovitskiy, A., Beyer, L., Kolesnikov, A., et al. (2021). An image is worth 16x16 words: Transformers for image recognition at scale. *ICLR 2021*.
